# 02 — Validação do Baseline, Feature Selection e Coerência Econômica

Projeto: **Pricing Optimization — Modelo de Simulação → Proposta**

Este notebook implementa um pipeline robusto para validar o baseline do modelo `fl_proposta`, incluindo:

1. split temporal OOT;
2. split temporal com purga por cliente/jornada;
3. forward selection por blocos de features;
4. métricas de modelo: ROC AUC, PR AUC, KS, Brier, Precision@K, Recall@K, Lift@K;
5. métricas de features: WoE, IV, KS individual, missing rate e PSI;
6. permutation importance no OOT;
7. SHAP, caso a biblioteca esteja disponível;
8. adversarial validation;
9. null importance;
10. rolling backtest temporal;
11. testes de coerência econômica: ranking por risco, monotonicidade por tier, sensibilidade à taxa e valor financiado capturado no Top K;
12. comparação opcional entre modelo livre e modelo com monotonic constraints.

> Ajuste caminhos, nomes de colunas e ordenação de risco conforme seu ambiente.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', 300)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 220)

RANDOM_STATE = 42

BASE_DIR = Path('.')
FEATURE_DIR = BASE_DIR / 'data' / 'features'
REPORT_DIR = BASE_DIR / 'outputs' / 'reports' / 'modelo_proposta'
MODEL_DIR = BASE_DIR / 'outputs' / 'models' / 'modelo_proposta'

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FEATURE_STORE = FEATURE_DIR / 'fs_simulacoes_proposta.parquet'


In [ ]:
try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
    print('LightGBM disponível.')
except Exception:
    HAS_LIGHTGBM = False
    print('LightGBM não disponível. Usando fallback sklearn.')

try:
    import shap
    HAS_SHAP = True
    print('SHAP disponível.')
except Exception:
    HAS_SHAP = False
    print('SHAP não disponível. Etapa SHAP será pulada.')

if not HAS_LIGHTGBM:
    from sklearn.ensemble import HistGradientBoostingClassifier


## 1. Carregamento da Feature Store

A entrada esperada é a feature store criada no notebook anterior:

```text
data/features/fs_simulacoes_proposta.parquet
```

Ela deve conter `target_proposta`, `dt_simulacao`, chaves de auditoria e features com prefixo `fsim_`.


In [ ]:
if 'fs_simulacoes' in globals():
    fs = fs_simulacoes.copy()
    print('Usando DataFrame existente: fs_simulacoes')
else:
    if not INPUT_FEATURE_STORE.exists():
        raise FileNotFoundError(
            f'Feature Store não encontrada: {INPUT_FEATURE_STORE}. '
            'Ajuste INPUT_FEATURE_STORE ou rode o notebook 01 antes.'
        )
    fs = pd.read_parquet(INPUT_FEATURE_STORE)
    print(f'Feature Store carregada de: {INPUT_FEATURE_STORE}')

fs['dt_simulacao'] = pd.to_datetime(fs['dt_simulacao'], errors='coerce')
fs['target_proposta'] = fs['target_proposta'].astype(int)

print(fs.shape)
display(fs.head())
display(fs['target_proposta'].value_counts(normalize=True).rename('target_rate'))


## 2. Definição de chaves, target e blocos de features

A seleção será por blocos para evitar um forward selection instável feature a feature.


In [ ]:
target_col = 'target_proposta'

key_cols = [
    'id_chave_execucao',
    'id_pricing',
    'id_cpf_cnpj',
    'id_jornada_proxy',
    'dt_simulacao',
    'periodo_arquivo',
]

base_safe_cols = [
    'id_cluster',
    'tp_politica',
    'tp_teste_elasticidade',
    'num_prazo',
    'cod_rev',
    'vl_financiado',
    'vl_entrada',
    'pct_entrada',
    'tp_gh',
    'fl_zero',
    'num_ano_mod_veic',
    'tx_simul',
    'tp_retorno_comissao',
    'fx_ano_mod_veic',
    'fx_entrada',
]

base_safe_cols = [c for c in base_safe_cols if c in fs.columns]
all_fsim_cols = sorted([c for c in fs.columns if c.startswith('fsim_')])

block_features = {
    'base_direta': base_safe_cols,
    'tempo': [c for c in all_fsim_cols if any(s in c for s in [
        'ano_simulacao', 'mes_simulacao', 'dia_mes', 'dia_semana',
        'semana_ano', 'trimestre', 'hora', 'inicio_mes', 'fim_mes', 'ano_mes'
    ])],
    'financeiro_simulacao': [c for c in all_fsim_cols if any(s in c for s in [
        'log_vl', 'pct_entrada_calc', 'gap_pct', 'por_prazo',
        'idade_veiculo', 'veiculo_0km', 'veiculo_9mais',
        'tx_x', 'tx_por_prazo'
    ])],
    'interacoes': [c for c in all_fsim_cols if 'inter_' in c],
    'cliente_quantidade': [c for c in all_fsim_cols if 'fsim_cli_' in c],
    'jornada': [c for c in all_fsim_cols if 'fsim_jorn_' in c],
    'preco_relativo_segmento': [c for c in all_fsim_cols if any(s in c for s in [
        'fsim_seg_', 'vs_media_passada', 'ratio_media_passada', 'abaixo_media_passada'
    ])],
    'missing_quality': [c for c in all_fsim_cols if 'missing' in c],
}

block_features = {k: sorted(list(set(v))) for k, v in block_features.items() if len(v) > 0}

for block, cols in block_features.items():
    print(f'{block}: {len(cols)} features')


## 3. Split temporal principal e split com purga

Split principal sugerido:

- treino: `202511` a `202603`;
- validação: `202604`;
- teste OOT: `202605` a `202606`.

A purga remove do treino clientes próximos à validação/teste para reduzir vazamento de jornadas em andamento.


In [ ]:
if 'ano_mes' not in fs.columns:
    fs['ano_mes'] = fs['dt_simulacao'].dt.strftime('%Y%m')

fs['split_temporal'] = np.select(
    [
        fs['ano_mes'].between('202511', '202603'),
        fs['ano_mes'].eq('202604'),
        fs['ano_mes'].between('202605', '202606'),
    ],
    ['train', 'valid', 'test'],
    default='out'
)

display(fs[['split_temporal', target_col]].value_counts(dropna=False).sort_index())
display(fs.groupby('split_temporal')[target_col].agg(['count', 'mean']).sort_index())


In [ ]:
def apply_purge_by_group(
    df,
    split_col='split_temporal',
    date_col='dt_simulacao',
    group_col='id_cpf_cnpj',
    train_label='train',
    future_labels=('valid', 'test'),
    purge_days=7,
):
    df = df.copy()
    df['split_purged'] = df[split_col].copy()
    future_df = df[df[split_col].isin(future_labels)][[group_col, date_col]].dropna()
    if future_df.empty or group_col not in df.columns:
        return df

    min_future_by_group = future_df.groupby(group_col)[date_col].min().rename('min_future_dt').reset_index()
    df = df.merge(min_future_by_group, on=group_col, how='left')

    purge_start = df['min_future_dt'] - pd.to_timedelta(purge_days, unit='D')
    mask_purge = (
        (df[split_col] == train_label)
        & df['min_future_dt'].notna()
        & (df[date_col] >= purge_start)
        & (df[date_col] < df['min_future_dt'])
    )
    df.loc[mask_purge, 'split_purged'] = 'purged_from_train'
    return df.drop(columns=['min_future_dt'])

fs = apply_purge_by_group(fs, group_col='id_cpf_cnpj', purge_days=7)
display(fs[['split_purged', target_col]].value_counts(dropna=False).sort_index())


## 4. Métricas auxiliares


In [ ]:
def safe_auc(metric_fn, y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return metric_fn(y_true, y_score)

def ks_score(y_true, y_score):
    df_ks = pd.DataFrame({'y': y_true, 'score': y_score}).sort_values('score', ascending=False)
    total_pos = df_ks['y'].sum()
    total_neg = len(df_ks) - total_pos
    if total_pos == 0 or total_neg == 0:
        return np.nan
    df_ks['cum_pos'] = df_ks['y'].cumsum() / total_pos
    df_ks['cum_neg'] = (1 - df_ks['y']).cumsum() / total_neg
    return (df_ks['cum_pos'] - df_ks['cum_neg']).abs().max()

def topk_metrics(y_true, y_score, value=None, ks=(0.01, 0.05, 0.10, 0.20)):
    out = {}
    data = pd.DataFrame({'y': np.asarray(y_true), 'score': np.asarray(y_score)})
    data['value'] = np.asarray(value) if value is not None else 1.0
    data = data.sort_values('score', ascending=False).reset_index(drop=True)
    base_rate = data['y'].mean()
    total_pos = data['y'].sum()
    total_value_pos = data.loc[data['y'] == 1, 'value'].sum()
    for k in ks:
        n_top = max(1, int(np.ceil(len(data) * k)))
        top = data.iloc[:n_top]
        precision = top['y'].mean()
        recall = top['y'].sum() / total_pos if total_pos > 0 else np.nan
        lift = precision / base_rate if base_rate > 0 else np.nan
        out[f'precision_at_{int(k*100)}pct'] = precision
        out[f'recall_at_{int(k*100)}pct'] = recall
        out[f'lift_at_{int(k*100)}pct'] = lift
        out[f'propostas_capturadas_at_{int(k*100)}pct'] = top['y'].sum()
        out[f'valor_pos_capturado_at_{int(k*100)}pct'] = (
            top.loc[top['y'] == 1, 'value'].sum() / total_value_pos if total_value_pos > 0 else np.nan
        )
    return out

def evaluate_model(y_true, y_score, value=None, prefix=''):
    metrics = {
        f'{prefix}n': len(y_true),
        f'{prefix}target_rate': np.mean(y_true),
        f'{prefix}roc_auc': safe_auc(roc_auc_score, y_true, y_score),
        f'{prefix}pr_auc': safe_auc(average_precision_score, y_true, y_score),
        f'{prefix}ks': ks_score(y_true, y_score),
        f'{prefix}brier': brier_score_loss(y_true, y_score),
    }
    metrics.update({f'{prefix}{k}': v for k, v in topk_metrics(y_true, y_score, value=value).items()})
    return metrics


## 5. Funções de treino

Usa LightGBM se disponível. Caso contrário, usa um fallback do scikit-learn com one-hot encoding.


In [ ]:
def infer_feature_types(df, features):
    categorical, numerical = [], []
    for col in features:
        if col not in df.columns:
            continue
        if df[col].dtype == 'object' or str(df[col].dtype).startswith('string') or str(df[col].dtype) == 'category':
            categorical.append(col)
        else:
            numerical.append(col)
    return numerical, categorical

def prepare_lgbm_frame(X, cat_cols):
    X = X.copy()
    for c in cat_cols:
        if c in X.columns:
            X[c] = X[c].astype('category')
    return X

def train_model(
    df,
    features,
    split_col='split_temporal',
    train_label='train',
    valid_label='valid',
    test_label='test',
    monotone_constraints=None,
    model_params=None,
):
    features = [c for c in features if c in df.columns]
    num_cols, cat_cols = infer_feature_types(df, features)

    train = df[df[split_col] == train_label].copy()
    valid = df[df[split_col] == valid_label].copy()
    test = df[df[split_col] == test_label].copy()

    X_train, y_train = train[features], train[target_col]
    X_valid, y_valid = valid[features], valid[target_col]
    X_test, y_test = test[features], test[target_col]

    if HAS_LIGHTGBM:
        params = dict(
            objective='binary',
            n_estimators=600,
            learning_rate=0.03,
            num_leaves=31,
            min_child_samples=100,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=1.0,
            reg_lambda=5.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )
        if model_params:
            params.update(model_params)
        if monotone_constraints is not None:
            params['monotone_constraints'] = monotone_constraints

        X_train_m = prepare_lgbm_frame(X_train, cat_cols)
        X_valid_m = prepare_lgbm_frame(X_valid, cat_cols)
        X_test_m = prepare_lgbm_frame(X_test, cat_cols)

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train_m,
            y_train,
            eval_set=[(X_valid_m, y_valid)],
            eval_metric='auc',
            categorical_feature=cat_cols if cat_cols else 'auto',
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        p_train = model.predict_proba(X_train_m)[:, 1]
        p_valid = model.predict_proba(X_valid_m)[:, 1]
        p_test = model.predict_proba(X_test_m)[:, 1]
    else:
        from sklearn.ensemble import HistGradientBoostingClassifier
        numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median'))])
        categorical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50)),
        ])
        preprocessor = ColumnTransformer([
            ('num', numeric_transformer, num_cols),
            ('cat', categorical_transformer, cat_cols),
        ])
        clf = HistGradientBoostingClassifier(
            learning_rate=0.04,
            max_iter=300,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        )
        model = Pipeline([('prep', preprocessor), ('clf', clf)])
        model.fit(X_train, y_train)
        p_train = model.predict_proba(X_train)[:, 1]
        p_valid = model.predict_proba(X_valid)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]

    metrics = {}
    metrics.update(evaluate_model(y_train, p_train, value=train.get('vl_financiado'), prefix='train_'))
    metrics.update(evaluate_model(y_valid, p_valid, value=valid.get('vl_financiado'), prefix='valid_'))
    metrics.update(evaluate_model(y_test, p_test, value=test.get('vl_financiado'), prefix='test_'))

    preds = {
        'train': pd.DataFrame({'index': train.index, 'y': y_train.values, 'score': p_train, 'split': train_label}),
        'valid': pd.DataFrame({'index': valid.index, 'y': y_valid.values, 'score': p_valid, 'split': valid_label}),
        'test': pd.DataFrame({'index': test.index, 'y': y_test.values, 'score': p_test, 'split': test_label}),
    }
    return model, metrics, preds, features, num_cols, cat_cols


## 6. Forward selection por blocos


In [ ]:
def run_block_experiments(df, block_features, split_col='split_temporal'):
    ordered_blocks = [
        'base_direta',
        'tempo',
        'financeiro_simulacao',
        'interacoes',
        'cliente_quantidade',
        'jornada',
        'preco_relativo_segmento',
        'missing_quality',
    ]
    results, models, preds_by_exp = [], {}, {}
    selected_features = []
    for i, block in enumerate(ordered_blocks):
        if block not in block_features:
            continue
        selected_features = sorted(list(set(selected_features + block_features[block])))
        exp_name = f'M{i:02d}_{block}'
        print(f'Treinando {exp_name} com {len(selected_features)} features...')
        model, metrics, preds, features_used, _, _ = train_model(df, selected_features, split_col=split_col)
        metrics['experiment'] = exp_name
        metrics['last_block_added'] = block
        metrics['n_features'] = len(features_used)
        results.append(metrics)
        models[exp_name] = model
        preds_by_exp[exp_name] = preds
    return pd.DataFrame(results), models, preds_by_exp

block_results, block_models, block_preds = run_block_experiments(fs, block_features, split_col='split_temporal')

cols_display = [
    'experiment', 'last_block_added', 'n_features',
    'valid_pr_auc', 'valid_roc_auc', 'valid_ks', 'valid_lift_at_10pct',
    'valid_precision_at_10pct', 'valid_recall_at_10pct', 'valid_valor_pos_capturado_at_10pct',
    'test_pr_auc', 'test_roc_auc', 'test_ks', 'test_lift_at_10pct',
    'test_precision_at_10pct', 'test_recall_at_10pct', 'test_valor_pos_capturado_at_10pct',
]
display(block_results[[c for c in cols_display if c in block_results.columns]])
block_results.to_csv(REPORT_DIR / '01_block_forward_selection_results.csv', index=False)


In [ ]:
selection_metric = 'valid_lift_at_10pct'
best_exp = block_results.sort_values(selection_metric, ascending=False).iloc[0]['experiment']
best_model = block_models[best_exp]
best_block = block_results.loc[block_results['experiment'] == best_exp, 'last_block_added'].iloc[0]

ordered_blocks = [
    'base_direta', 'tempo', 'financeiro_simulacao', 'interacoes',
    'cliente_quantidade', 'jornada', 'preco_relativo_segmento', 'missing_quality'
]

best_features = []
for block in ordered_blocks:
    if block in block_features:
        best_features = sorted(list(set(best_features + block_features[block])))
    if block == best_block:
        break

print('Melhor experimento:', best_exp)
print('Métrica de seleção:', selection_metric)
print('Qtd features:', len(best_features))


## 7. WoE, IV e KS individual

Uso:
- detectar features com poder individual;
- investigar features com IV alto demais;
- avaliar bins e monotonicidade.


In [ ]:
def make_bins(s, max_bins=10):
    if pd.api.types.is_numeric_dtype(s):
        try:
            return pd.qcut(s, q=max_bins, duplicates='drop')
        except Exception:
            try:
                return pd.cut(s, bins=min(max_bins, max(2, s.nunique())), duplicates='drop')
            except Exception:
                return s.astype('string').fillna('MISSING')
    vc = s.astype('string').fillna('MISSING').value_counts(dropna=False)
    top = vc.head(max_bins - 1).index
    return s.astype('string').fillna('MISSING').where(s.astype('string').fillna('MISSING').isin(top), 'OTHER')

def woe_iv_table(df, feature, target, max_bins=10, eps=0.5):
    temp = df[[feature, target]].copy()
    temp['bin'] = make_bins(temp[feature], max_bins=max_bins).astype('string').fillna('MISSING')
    g = temp.groupby('bin', dropna=False)[target].agg(['count', 'sum']).rename(columns={'sum': 'event'})
    g['non_event'] = g['count'] - g['event']
    total_event = g['event'].sum()
    total_non_event = g['non_event'].sum()
    g['event_rate'] = g['event'] / g['count'].replace(0, np.nan)
    g['dist_event'] = (g['event'] + eps) / (total_event + eps * len(g))
    g['dist_non_event'] = (g['non_event'] + eps) / (total_non_event + eps * len(g))
    g['woe'] = np.log(g['dist_event'] / g['dist_non_event'])
    g['iv_component'] = (g['dist_event'] - g['dist_non_event']) * g['woe']
    g['feature'] = feature
    return g.reset_index()

def iv_ks_feature_summary(df, features, target, max_bins=10):
    rows, tables = [], []
    for feature in features:
        try:
            wt = woe_iv_table(df, feature, target, max_bins=max_bins)
            tables.append(wt)
            iv = wt['iv_component'].sum()
            temp = df[[feature, target]].copy()
            if pd.api.types.is_numeric_dtype(temp[feature]):
                score = temp[feature].fillna(temp[feature].median())
            else:
                rate_map = temp.groupby(feature, dropna=False)[target].mean()
                score = temp[feature].map(rate_map).fillna(temp[target].mean())
            rows.append({
                'feature': feature,
                'iv': iv,
                'ks_feature': ks_score(temp[target].values, score.values),
                'missing_rate': df[feature].isna().mean(),
                'n_unique': df[feature].nunique(dropna=True),
            })
        except Exception as e:
            rows.append({'feature': feature, 'iv': np.nan, 'ks_feature': np.nan, 'missing_rate': np.nan, 'n_unique': np.nan, 'error': str(e)})
    summary = pd.DataFrame(rows).sort_values('iv', ascending=False)
    all_tables = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
    return summary, all_tables

train_df = fs[fs['split_temporal'] == 'train'].copy()
valid_df = fs[fs['split_temporal'] == 'valid'].copy()
test_df = fs[fs['split_temporal'] == 'test'].copy()

iv_summary, woe_tables = iv_ks_feature_summary(train_df, best_features, target_col, max_bins=10)
display(iv_summary.head(50))
iv_summary.to_csv(REPORT_DIR / '02_iv_ks_feature_summary_train.csv', index=False)
woe_tables.to_csv(REPORT_DIR / '03_woe_tables_train.csv', index=False)


## 8. PSI — estabilidade temporal das features


In [ ]:
def psi_between(expected, actual, bins=10, eps=1e-6):
    expected = pd.Series(expected)
    actual = pd.Series(actual)
    if pd.api.types.is_numeric_dtype(expected):
        try:
            cats = pd.qcut(expected.dropna(), q=bins, duplicates='drop').cat.categories
            exp_bins = pd.cut(expected, bins=cats)
            act_bins = pd.cut(actual, bins=cats)
        except Exception:
            combined = pd.concat([expected, actual]).dropna()
            if combined.nunique() <= 1:
                return 0.0
            cuts = np.linspace(combined.min(), combined.max(), min(bins, combined.nunique()) + 1)
            exp_bins = pd.cut(expected, bins=cuts, include_lowest=True)
            act_bins = pd.cut(actual, bins=cuts, include_lowest=True)
    else:
        top = expected.astype('string').fillna('MISSING').value_counts().head(bins - 1).index
        exp_bins = expected.astype('string').fillna('MISSING').where(expected.astype('string').fillna('MISSING').isin(top), 'OTHER')
        act_bins = actual.astype('string').fillna('MISSING').where(actual.astype('string').fillna('MISSING').isin(top), 'OTHER')

    exp_dist = exp_bins.astype('string').fillna('MISSING').value_counts(normalize=True)
    act_dist = act_bins.astype('string').fillna('MISSING').value_counts(normalize=True)
    idx = exp_dist.index.union(act_dist.index)
    exp_p = exp_dist.reindex(idx, fill_value=eps).clip(lower=eps)
    act_p = act_dist.reindex(idx, fill_value=eps).clip(lower=eps)
    return ((act_p - exp_p) * np.log(act_p / exp_p)).sum()

def psi_report(train, valid, test, features):
    rows = []
    for feature in features:
        try:
            psi_tv = psi_between(train[feature], valid[feature])
            psi_tt = psi_between(train[feature], test[feature])
            max_psi = max(psi_tv, psi_tt)
            rows.append({
                'feature': feature,
                'psi_train_valid': psi_tv,
                'psi_train_test': psi_tt,
                'max_psi': max_psi,
                'status': 'OK' if max_psi < 0.10 else ('ATENCAO' if max_psi < 0.25 else 'DRIFT_RELEVANTE'),
            })
        except Exception as e:
            rows.append({'feature': feature, 'psi_train_valid': np.nan, 'psi_train_test': np.nan, 'max_psi': np.nan, 'status': 'ERRO', 'error': str(e)})
    return pd.DataFrame(rows).sort_values('max_psi', ascending=False)

psi_df = psi_report(train_df, valid_df, test_df, best_features)
display(psi_df.head(50))
psi_df.to_csv(REPORT_DIR / '04_psi_feature_stability.csv', index=False)


## 9. Permutation importance no OOT


In [ ]:
def predict_proba_model(model, X, cat_cols):
    Xp = X.copy()
    if HAS_LIGHTGBM:
        for c in cat_cols:
            if c in Xp.columns:
                Xp[c] = Xp[c].astype('category')
    return model.predict_proba(Xp)[:, 1]

X_test = test_df[best_features].copy()
y_test = test_df[target_col].copy()
_, best_cat_cols = infer_feature_types(fs, best_features)

if HAS_LIGHTGBM and len(X_test) > 0:
    sample_size = min(100_000, len(X_test))
    sample_idx = X_test.sample(sample_size, random_state=RANDOM_STATE).index
    X_perm = X_test.loc[sample_idx].copy()
    y_perm = y_test.loc[sample_idx].copy()
    for c in best_cat_cols:
        if c in X_perm.columns:
            X_perm[c] = X_perm[c].astype('category')
    perm = permutation_importance(
        best_model,
        X_perm,
        y_perm,
        n_repeats=5,
        random_state=RANDOM_STATE,
        scoring='average_precision',
        n_jobs=-1,
    )
    perm_df = pd.DataFrame({
        'feature': best_features,
        'perm_importance_mean': perm.importances_mean,
        'perm_importance_std': perm.importances_std,
    }).sort_values('perm_importance_mean', ascending=False)
    display(perm_df.head(50))
    perm_df.to_csv(REPORT_DIR / '05_permutation_importance_test.csv', index=False)
else:
    print('Permutation importance pulada.')


## 10. SHAP no OOT, se disponível


In [ ]:
if HAS_SHAP and HAS_LIGHTGBM and len(X_test) > 0:
    shap_sample_size = min(5000, len(X_test))
    shap_idx = X_test.sample(shap_sample_size, random_state=RANDOM_STATE).index
    X_shap = X_test.loc[shap_idx].copy()
    for c in best_cat_cols:
        if c in X_shap.columns:
            X_shap[c] = X_shap[c].astype('category')
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_shap)
    shap_values_pos = shap_values[1] if isinstance(shap_values, list) else shap_values
    shap_importance = pd.DataFrame({
        'feature': best_features,
        'shap_mean_abs': np.abs(shap_values_pos).mean(axis=0),
    }).sort_values('shap_mean_abs', ascending=False)
    display(shap_importance.head(50))
    shap_importance.to_csv(REPORT_DIR / '06_shap_importance_test.csv', index=False)
else:
    print('SHAP pulado.')


## 11. Adversarial validation — treino vs teste

Se a AUC for alta, treino e teste são muito distinguíveis, indicando drift de população, política ou canal.


In [ ]:
def adversarial_validation(df, features, left_label='train', right_label='test'):
    if not HAS_LIGHTGBM:
        return np.nan, pd.DataFrame()
    adv = df[df['split_temporal'].isin([left_label, right_label])].copy()
    adv['adv_target'] = (adv['split_temporal'] == right_label).astype(int)
    n_min = adv['adv_target'].value_counts().min()
    adv_sample = (
        adv.groupby('adv_target', group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n_min), random_state=RANDOM_STATE))
        .sample(frac=1, random_state=RANDOM_STATE)
    )
    train_adv, valid_adv = train_test_split(
        adv_sample,
        test_size=0.3,
        random_state=RANDOM_STATE,
        stratify=adv_sample['adv_target'],
    )
    _, cat_cols = infer_feature_types(adv_sample, features)
    X_tr = prepare_lgbm_frame(train_adv[features], cat_cols)
    X_va = prepare_lgbm_frame(valid_adv[features], cat_cols)
    model = lgb.LGBMClassifier(
        objective='binary', n_estimators=300, learning_rate=0.05,
        num_leaves=31, min_child_samples=100, subsample=0.8,
        colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    )
    model.fit(X_tr, train_adv['adv_target'])
    pred = model.predict_proba(X_va)[:, 1]
    auc = roc_auc_score(valid_adv['adv_target'], pred)
    imp = pd.DataFrame({'feature': features, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
    return auc, imp

adv_auc, adv_imp = adversarial_validation(fs, best_features)
print('Adversarial AUC treino vs teste:', adv_auc)
display(adv_imp.head(30))
adv_imp.to_csv(REPORT_DIR / '07_adversarial_validation_importance.csv', index=False)


## 12. Null importance

Compara importância real contra importância obtida com target embaralhado.


In [ ]:
def null_importance(df, features, n_runs=5, sample_size=200_000):
    if not HAS_LIGHTGBM:
        print('Null importance implementada apenas para LightGBM.')
        return pd.DataFrame()
    base = df[df['split_temporal'].isin(['train', 'valid'])].copy()
    if len(base) > sample_size:
        base = base.sample(sample_size, random_state=RANDOM_STATE)
    train = base[base['split_temporal'] == 'train'].copy()
    _, cat_cols = infer_feature_types(base, features)
    X_train = prepare_lgbm_frame(train[features], cat_cols)
    real_model = lgb.LGBMClassifier(
        objective='binary', n_estimators=300, learning_rate=0.04,
        num_leaves=31, min_child_samples=100, subsample=0.8,
        colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    )
    real_model.fit(X_train, train[target_col])
    real_imp = pd.Series(real_model.feature_importances_, index=features)
    null_imps = []
    for i in range(n_runs):
        y_null = train[target_col].sample(frac=1, random_state=RANDOM_STATE + i).values
        null_model = lgb.LGBMClassifier(
            objective='binary', n_estimators=300, learning_rate=0.04,
            num_leaves=31, min_child_samples=100, subsample=0.8,
            colsample_bytree=0.8, random_state=RANDOM_STATE + i, n_jobs=-1, verbose=-1,
        )
        null_model.fit(X_train, y_null)
        null_imps.append(pd.Series(null_model.feature_importances_, index=features))
    null_imp_df = pd.concat(null_imps, axis=1)
    out = pd.DataFrame({
        'feature': features,
        'real_importance': real_imp.values,
        'null_mean': null_imp_df.mean(axis=1).values,
        'null_p95': null_imp_df.quantile(0.95, axis=1).values,
    })
    out['importance_above_null_p95'] = out['real_importance'] > out['null_p95']
    out['real_minus_null_p95'] = out['real_importance'] - out['null_p95']
    return out.sort_values('real_minus_null_p95', ascending=False)

null_imp_df = null_importance(fs, best_features, n_runs=5)
display(null_imp_df.head(50))
null_imp_df.to_csv(REPORT_DIR / '08_null_importance.csv', index=False)


## 13. Score dataframe e calibração por decil


In [ ]:
def score_dataframe_for_exp(df, preds, split_name='test'):
    p = preds[split_name].copy()
    out = df.loc[p['index']].copy()
    out['score'] = p['score'].values
    out['y_true'] = p['y'].values
    return out

def decile_report(df, score_col='score', target_col='y_true', value_col='vl_financiado', n_bins=10):
    d = df.copy()
    d['score_decile'] = pd.qcut(d[score_col].rank(method='first'), q=n_bins, labels=False) + 1
    d['score_decile'] = n_bins + 1 - d['score_decile']
    agg = d.groupby('score_decile').agg(
        n=(target_col, 'size'),
        score_min=(score_col, 'min'),
        score_max=(score_col, 'max'),
        score_mean=(score_col, 'mean'),
        event_rate=(target_col, 'mean'),
        events=(target_col, 'sum'),
        vl_financiado_total=(value_col, 'sum') if value_col in d.columns else (target_col, 'size'),
        vl_financiado_eventos=(value_col, lambda s: s[d.loc[s.index, target_col] == 1].sum()) if value_col in d.columns else (target_col, 'sum'),
    ).reset_index()
    agg['lift_vs_base'] = agg['event_rate'] / d[target_col].mean()
    return agg

best_pred_df_test = score_dataframe_for_exp(fs, block_preds[best_exp], split_name='test')
best_pred_df_valid = score_dataframe_for_exp(fs, block_preds[best_exp], split_name='valid')

decile_test = decile_report(best_pred_df_test)
display(decile_test)
decile_test.to_csv(REPORT_DIR / '09_calibration_decile_test.csv', index=False)


## 14. Coerência econômica por tier de risco

Aqui avaliamos score médio e taxa real por `tp_gh`.


In [ ]:
def segment_score_report(df, segment_col, score_col='score', target_col='y_true', value_col='vl_financiado'):
    if segment_col not in df.columns:
        return pd.DataFrame()
    agg = df.groupby(segment_col).agg(
        n=(target_col, 'size'),
        score_mean=(score_col, 'mean'),
        score_median=(score_col, 'median'),
        event_rate=(target_col, 'mean'),
        events=(target_col, 'sum'),
        vl_financiado_mean=(value_col, 'mean') if value_col in df.columns else (target_col, 'size'),
        vl_financiado_total=(value_col, 'sum') if value_col in df.columns else (target_col, 'size'),
    ).reset_index()
    return agg.sort_values('score_mean', ascending=False)

gh_report_test = segment_score_report(best_pred_df_test, 'tp_gh')
display(gh_report_test)
gh_report_test.to_csv(REPORT_DIR / '10_segment_score_report_tp_gh_test.csv', index=False)


## 15. Teste de monotonicidade por risco controlando condições comerciais

A premissa econômica é:

> Mantidas condições comerciais parecidas, clientes de melhor risco devem tender a ter score maior.

Ajuste `risk_order_map` conforme o domínio real.


In [ ]:
risk_order_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'H': 8, 'I': 9, 'I+': 10}

def normalize_gh(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().upper()

def risk_monotonicity_report(
    df,
    risk_col='tp_gh',
    score_col='score',
    min_cell_n=100,
    control_cols=('fx_entrada', 'fx_ano_mod_veic', 'num_prazo', 'tp_politica'),
):
    d = df.copy()
    if risk_col not in d.columns:
        return pd.DataFrame()
    d['risk_ord'] = d[risk_col].map(lambda x: risk_order_map.get(normalize_gh(x), np.nan))
    d = d.dropna(subset=['risk_ord'])
    control_cols = [c for c in control_cols if c in d.columns]
    rows = []
    for keys, g in d.groupby(control_cols):
        if len(g) < min_cell_n:
            continue
        by_risk = g.groupby('risk_ord')[score_col].mean().sort_index()
        if len(by_risk) < 2:
            continue
        diffs = by_risk.diff().dropna()
        n_viol = (diffs > 0).sum()
        n_comp = len(diffs)
        rows.append({
            'control_cell': keys,
            'n': len(g),
            'n_risk_levels': len(by_risk),
            'n_comparisons': n_comp,
            'n_violations': n_viol,
            'violation_rate': n_viol / n_comp if n_comp > 0 else np.nan,
            'score_by_risk': by_risk.to_dict(),
        })
    out = pd.DataFrame(rows)
    return out.sort_values('violation_rate', ascending=False) if len(out) else out

mono_report = risk_monotonicity_report(best_pred_df_test)
display(mono_report.head(30))
print('Taxa média de violação:', mono_report['violation_rate'].mean() if len(mono_report) else np.nan)
mono_report.to_csv(REPORT_DIR / '11_risk_monotonicity_report_test.csv', index=False)


## 16. Teste contrafactual de sensibilidade à taxa

Mantém as demais variáveis constantes e altera `tx_simul`.

Expectativa: taxa menor tende a aumentar score; taxa maior tende a reduzir score.

Este teste é um sanity check, não uma prova causal.


In [ ]:
def counterfactual_tax_test(df, model, features, cat_cols, sample_size=10000, delta=0.10, tax_col='tx_simul'):
    if tax_col not in features:
        print(f'{tax_col} não está nas features. Teste pulado.')
        return pd.DataFrame()
    test_base = df[df['split_temporal'] == 'test'].copy()
    sample = test_base.sample(min(sample_size, len(test_base)), random_state=RANDOM_STATE).copy()
    X_base = sample[features].copy()
    X_down = X_base.copy()
    X_up = X_base.copy()
    X_down[tax_col] = X_down[tax_col] - delta
    X_up[tax_col] = X_up[tax_col] + delta
    p_base = predict_proba_model(model, X_base, cat_cols)
    p_down = predict_proba_model(model, X_down, cat_cols)
    p_up = predict_proba_model(model, X_up, cat_cols)
    out_cols = [c for c in ['id_chave_execucao', 'id_cpf_cnpj', 'dt_simulacao', 'tp_gh', 'tx_simul', target_col] if c in sample.columns]
    out = sample[out_cols].copy()
    out['score_base'] = p_base
    out['score_taxa_menor'] = p_down
    out['score_taxa_maior'] = p_up
    out['delta_score_taxa_menor'] = out['score_taxa_menor'] - out['score_base']
    out['delta_score_taxa_maior'] = out['score_taxa_maior'] - out['score_base']
    out['taxa_menor_aumentou_score'] = out['delta_score_taxa_menor'] > 0
    out['taxa_maior_reduziu_score'] = out['delta_score_taxa_maior'] < 0
    return out

cf_tax = counterfactual_tax_test(fs, best_model, best_features, best_cat_cols, delta=0.10)
display(cf_tax.head())
print('Taxa menor aumentou score em % dos casos:', cf_tax['taxa_menor_aumentou_score'].mean() if len(cf_tax) else np.nan)
print('Taxa maior reduziu score em % dos casos:', cf_tax['taxa_maior_reduziu_score'].mean() if len(cf_tax) else np.nan)
cf_tax.to_csv(REPORT_DIR / '12_counterfactual_tax_sensitivity_test.csv', index=False)


## 17. Métricas econômicas no Top K

Avalia se o topo do ranking captura propostas e valor financiado.


In [ ]:
def economic_topk_report(df, score_col='score', target_col='y_true', value_col='vl_financiado', risk_col='tp_gh', ks=(0.01, 0.05, 0.10, 0.20)):
    d = df.sort_values(score_col, ascending=False).copy()
    total_events = d[target_col].sum()
    total_event_value = d.loc[d[target_col] == 1, value_col].sum() if value_col in d.columns else np.nan
    rows = []
    for k in ks:
        n_top = max(1, int(np.ceil(len(d) * k)))
        top = d.iloc[:n_top]
        row = {
            'top_pct': k,
            'n_top': n_top,
            'event_rate_top': top[target_col].mean(),
            'events_top': top[target_col].sum(),
            'recall_events_top': top[target_col].sum() / total_events if total_events > 0 else np.nan,
            'vl_financiado_total_top': top[value_col].sum() if value_col in top.columns else np.nan,
            'vl_financiado_medio_top': top[value_col].mean() if value_col in top.columns else np.nan,
            'vl_financiado_eventos_top': top.loc[top[target_col] == 1, value_col].sum() if value_col in top.columns else np.nan,
            'share_vl_financiado_eventos_top': top.loc[top[target_col] == 1, value_col].sum() / total_event_value if value_col in top.columns and total_event_value > 0 else np.nan,
        }
        if risk_col in top.columns:
            for rk, val in top[risk_col].value_counts(normalize=True).to_dict().items():
                row[f'mix_risk_{rk}'] = val
        rows.append(row)
    return pd.DataFrame(rows)

economic_topk = economic_topk_report(best_pred_df_test)
display(economic_topk)
economic_topk.to_csv(REPORT_DIR / '13_economic_topk_report_test.csv', index=False)


## 18. Rolling backtest temporal

Mede estabilidade em múltiplas janelas temporais.


In [ ]:
def rolling_backtest(df, features, periods, min_train_periods=2):
    results = []
    periods = sorted(periods)
    for i in range(min_train_periods, len(periods)):
        train_periods = periods[:i]
        valid_period = periods[i]
        temp = df[df['ano_mes'].isin(train_periods + [valid_period])].copy()
        temp['split_roll'] = np.where(temp['ano_mes'].isin(train_periods), 'train', 'valid')
        temp_valid_as_test = temp[temp['split_roll'] == 'valid'].copy()
        temp_valid_as_test['split_roll'] = 'test'
        temp_model = pd.concat([temp, temp_valid_as_test], ignore_index=True)
        try:
            _, metrics, _, _, _, _ = train_model(
                temp_model,
                features=features,
                split_col='split_roll',
                train_label='train',
                valid_label='valid',
                test_label='test',
            )
            metrics['train_period_start'] = train_periods[0]
            metrics['train_period_end'] = train_periods[-1]
            metrics['valid_period'] = valid_period
            results.append(metrics)
            print(f'OK: treino até {train_periods[-1]} | valida {valid_period}')
        except Exception as e:
            print(f'Erro em {valid_period}: {e}')
    return pd.DataFrame(results)

available_periods = sorted(fs.loc[fs['split_temporal'].isin(['train', 'valid', 'test']), 'ano_mes'].dropna().unique())
print('Períodos disponíveis:', available_periods)

rolling_results = rolling_backtest(fs, best_features, periods=available_periods, min_train_periods=2)
if len(rolling_results):
    display(rolling_results[[
        'train_period_start', 'train_period_end', 'valid_period',
        'valid_pr_auc', 'valid_roc_auc', 'valid_ks',
        'valid_lift_at_10pct', 'valid_precision_at_10pct', 'valid_recall_at_10pct',
    ]])
else:
    display(rolling_results)
rolling_results.to_csv(REPORT_DIR / '14_rolling_backtest_results.csv', index=False)


## 19. Modelo com monotonic constraints — opcional

Testa restrições em:
- `tx_simul`: maior taxa tende a reduzir score;
- `tp_gh_ordinal`: maior valor = pior risco, tende a reduzir score.

Compare performance com coerência econômica.


In [ ]:
fs_mono = fs.copy()
if 'tp_gh' in fs_mono.columns:
    fs_mono['tp_gh_norm'] = fs_mono['tp_gh'].map(normalize_gh)
    fs_mono['tp_gh_ordinal'] = fs_mono['tp_gh_norm'].map(risk_order_map)

mono_features = sorted(list(set(best_features + ['tp_gh_ordinal'])))
mono_features = [c for c in mono_features if c in fs_mono.columns]

if HAS_LIGHTGBM:
    mono_constraints = []
    for f in mono_features:
        if f == 'tx_simul':
            mono_constraints.append(-1)
        elif f == 'tp_gh_ordinal':
            mono_constraints.append(-1)
        else:
            mono_constraints.append(0)
    mono_model, mono_metrics, mono_preds, _, _, _ = train_model(
        fs_mono,
        features=mono_features,
        split_col='split_temporal',
        monotone_constraints=mono_constraints,
    )
    free_metrics = block_results.loc[block_results['experiment'] == best_exp].iloc[0].to_dict()
    mono_compare = pd.DataFrame([
        {'model': 'livre', **free_metrics},
        {'model': 'monotonic_constraints', **mono_metrics},
    ])
    display(mono_compare[[
        'model',
        'valid_pr_auc', 'valid_roc_auc', 'valid_ks', 'valid_lift_at_10pct',
        'test_pr_auc', 'test_roc_auc', 'test_ks', 'test_lift_at_10pct',
    ]])
    mono_compare.to_csv(REPORT_DIR / '15_model_free_vs_monotonic_constraints.csv', index=False)
else:
    print('Monotonic constraints pulado: LightGBM indisponível.')


## 20. Re-ranking econômico

Compara ranking puro por `P(proposta)` versus ranking econômico:

```text
score_economico = P(proposta) × valor_financiado × fator_risco
```

Ajuste `risk_factor_map` com o time de negócio/risco.


In [ ]:
risk_factor_map = {'A': 1.20, 'B': 1.15, 'C': 1.05, 'D': 1.00, 'E': 0.95, 'F': 0.90, 'G': 0.80, 'H': 0.70, 'I': 0.60, 'I+': 0.55}

rerank_df = best_pred_df_test.copy()
if 'tp_gh' in rerank_df.columns:
    rerank_df['risk_factor'] = rerank_df['tp_gh'].map(lambda x: risk_factor_map.get(normalize_gh(x), 1.0))
else:
    rerank_df['risk_factor'] = 1.0

rerank_df['score_economico'] = (
    rerank_df['score']
    * rerank_df['vl_financiado'].fillna(rerank_df['vl_financiado'].median())
    * rerank_df['risk_factor']
)

pure_topk = economic_topk_report(rerank_df, score_col='score')
econ_topk = economic_topk_report(rerank_df, score_col='score_economico')
pure_topk['ranking'] = 'score_puro'
econ_topk['ranking'] = 'score_economico'
rerank_compare = pd.concat([pure_topk, econ_topk], ignore_index=True)
display(rerank_compare)
rerank_compare.to_csv(REPORT_DIR / '16_reranking_economico_comparison.csv', index=False)


## 21. Relatório consolidado


In [ ]:
summary = {
    'best_experiment': best_exp,
    'selection_metric': selection_metric,
    'n_best_features': len(best_features),
    'valid_pr_auc': float(block_results.loc[block_results['experiment'] == best_exp, 'valid_pr_auc'].iloc[0]),
    'valid_roc_auc': float(block_results.loc[block_results['experiment'] == best_exp, 'valid_roc_auc'].iloc[0]),
    'valid_ks': float(block_results.loc[block_results['experiment'] == best_exp, 'valid_ks'].iloc[0]),
    'valid_lift_at_10pct': float(block_results.loc[block_results['experiment'] == best_exp, 'valid_lift_at_10pct'].iloc[0]),
    'test_pr_auc': float(block_results.loc[block_results['experiment'] == best_exp, 'test_pr_auc'].iloc[0]),
    'test_roc_auc': float(block_results.loc[block_results['experiment'] == best_exp, 'test_roc_auc'].iloc[0]),
    'test_ks': float(block_results.loc[block_results['experiment'] == best_exp, 'test_ks'].iloc[0]),
    'test_lift_at_10pct': float(block_results.loc[block_results['experiment'] == best_exp, 'test_lift_at_10pct'].iloc[0]),
    'adversarial_auc_train_vs_test': float(adv_auc) if adv_auc == adv_auc else None,
    'risk_monotonicity_violation_rate_mean': float(mono_report['violation_rate'].mean()) if len(mono_report) else None,
    'taxa_menor_aumentou_score_pct': float(cf_tax['taxa_menor_aumentou_score'].mean()) if len(cf_tax) else None,
    'taxa_maior_reduziu_score_pct': float(cf_tax['taxa_maior_reduziu_score'].mean()) if len(cf_tax) else None,
}

summary_df = pd.DataFrame([summary])
display(summary_df)
summary_df.to_csv(REPORT_DIR / '00_summary_modelo_proposta.csv', index=False)
with open(REPORT_DIR / '00_summary_modelo_proposta.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f'Relatórios salvos em: {REPORT_DIR}')


## 22. Checklist antes da próxima feature store

Antes de avançar para target encoding temporal, conversão histórica por revenda/cluster/política ou elasticidade segmentada, valide:

1. O melhor bloco melhora também no OOT, não apenas na validação.
2. As principais features têm PSI aceitável.
3. Features com IV muito alto foram investigadas para leakage.
4. O ranking por `tp_gh` faz sentido controlando taxa, entrada, prazo e ticket.
5. O teste contrafactual de taxa tem comportamento plausível.
6. O modelo livre foi comparado com o modelo monotônico.
7. A métrica de decisão do negócio foi definida:
   - `P(proposta)`;
   - `P(proposta) × valor_financiado`;
   - `P(proposta) × valor_financiado × fator_risco`;
   - ou outra função econômica definida pelo time de pricing/risco.
